# 08 · Fabric — Step 01: Load the Lakehouse (PySpark)
**Run this notebook inside Microsoft Fabric, attached to a Lakehouse** (it uses the Fabric-provided `spark` session). It loads the exported Parquet as managed Delta tables and creates the convenience views the Data Agent and semantic model query.

**Before running:**
1. Run notebooks 01–07 locally so `output/tables/` and `output/gold/` exist.
2. Upload `output/tables/*.parquet` to the Lakehouse Files area under `regimpact_raw/`.
3. Upload `output/gold/*.parquet` under `regimpact_gold/`.
4. Attach this notebook to the Lakehouse and Run all.

**What the next cell does:** sets the Lakehouse Files folders where you uploaded the Parquet files.

In [ ]:
RAW_FOLDER = 'Files/regimpact_raw'    # uploaded entity parquet files
GOLD_FOLDER = 'Files/regimpact_gold'  # uploaded Gold star-schema files

**What the next cell does:** loads each raw entity Parquet file as a managed Delta table.

In [ ]:
TABLES = [
    'regulations', 'regulatory_changes', 'obligations', 'controls', 'capabilities',
    'technologies', 'evidence', 'systems', 'business_processes', 'products',
    'data_domains', 'business_units', 'risks', 'gaps', 'remediation_actions',
    'compliance_scores', 'relationships',
]

for name in TABLES:
    path = f'{RAW_FOLDER}/{name}.parquet'
    try:
        df = spark.read.parquet(path)  # noqa: F821 (spark provided by Fabric)
        df.write.mode('overwrite').option('overwriteSchema', 'true').format('delta').saveAsTable(name)
        print(f'Loaded {name}: {df.count()} rows')
    except Exception as exc:
        print(f'Skipped {name}: {exc}')

**What the next cell does:** loads the Gold star-schema tables (the `dim_*` / `fact_*` tables the Power BI semantic model binds to).

In [ ]:
GOLD_TABLES = [
    'dim_regulation', 'dim_change', 'dim_obligation', 'dim_control',
    'dim_capability', 'dim_technology', 'dim_evidence', 'dim_system',
    'dim_process', 'dim_product', 'dim_data_domain', 'dim_unit', 'dim_risk',
    'fact_compliance_score', 'fact_gap', 'fact_remediation', 'bridge_gap_entity',
]

for name in GOLD_TABLES:
    path = f'{GOLD_FOLDER}/{name}.parquet'
    try:
        df = spark.read.parquet(path)  # noqa: F821
        df.write.mode('overwrite').option('overwriteSchema', 'true').format('delta').saveAsTable(name)
        print(f'Loaded {name}: {df.count()} rows')
    except Exception as exc:
        print(f'Skipped {name}: {exc}')

**What the next cell does:** creates `v_impact` — a denormalised view joining changes → obligations → gaps → remediation so the Data Agent can answer most questions from one place.

In [ ]:
spark.sql(  # noqa: F821
    '''
    CREATE OR REPLACE VIEW v_impact AS
    SELECT
        c.id AS change_id, c.title AS change_title, c.regulation_id,
        c.criticality AS change_criticality, c.effective_date,
        o.id AS obligation_id, o.statement AS obligation, o.theme, o.target_maturity,
        g.id AS gap_id, g.severity AS gap_severity, g.maturity_shortfall, g.control_id,
        ctl.capability_id, cap.name AS capability, g.rationale,
        r.action AS remediation, r.action_type, r.estimated_effort_days,
        r.priority AS remediation_priority, r.target_unit_id
    FROM regulatory_changes c
    JOIN obligations o ON o.change_id = c.id
    LEFT JOIN gaps g ON g.obligation_id = o.id
    LEFT JOIN remediation_actions r ON r.gap_id = g.id
    LEFT JOIN controls ctl ON ctl.id = g.control_id
    LEFT JOIN capabilities cap ON cap.id = ctl.capability_id
    '''
)
print('Created view v_impact')

**What the next cell does:** creates `v_compliance` — the before / after-change / after-remediation score per change.

In [ ]:
spark.sql(  # noqa: F821
    '''
    CREATE OR REPLACE VIEW v_compliance AS
    SELECT s.change_id, s.scope_type, s.scope_id, s.scope_name,
           s.scenario, s.score, s.status
    FROM compliance_scores s
    '''
)
print('Created view v_compliance')

**What the next cell does:** creates `v_capability_health` — control maturity + enabling technology + proving evidence (Layers 3, 4 and 5) in one place.

In [ ]:
spark.sql(  # noqa: F821
    '''
    CREATE OR REPLACE VIEW v_capability_health AS
    SELECT cap.id AS capability_id, cap.name AS capability, cap.domain,
           ctl.id AS control_id, ctl.name AS control, ctl.maturity,
           ctl.status AS control_status, ev.evidence_type,
           ev.status AS evidence_status, tec.name AS technology, tec.is_microsoft
    FROM capabilities cap
    LEFT JOIN controls ctl ON ctl.capability_id = cap.id
    LEFT JOIN evidence ev ON ev.control_id = ctl.id
    LEFT JOIN technologies tec ON tec.id = ev.technology_id
    '''
)
print('Created view v_capability_health')